In [ ]:
import torch
from torch import nntorch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


# Dataload

In [9]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

In [10]:
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens

print(f"vocab ({train_data.vocab_size} tokens): {train_data.vocab}")
print(f"families: {train_data.fam_to_idx}")
print(f"padding is ignored in the loss with ignore_index={IGNORE_INDEX}")

sequence = train_data.df["sequence"][0]
tokens = train_data.encode(sequence)
print()
print(f"encode / decode round-trips: {train_data.decode(tokens) == sequence}")
print(f"and from the shifted target too:  {train_data.decode(x_shifted) == sequence}")

vocab (23 tokens): {'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, 'A': 3, 'C': 4, 'D': 5, 'E': 6, 'F': 7, 'G': 8, 'H': 9, 'I': 10, 'K': 11, 'L': 12, 'M': 13, 'N': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'V': 20, 'W': 21, 'Y': 22}
families: {'AcyP': 0, 'HPr': 1, 'HU': 2}
padding is ignored in the loss with ignore_index=-1

encode / decode round-trips: True
and from the shifted target too:  True


# Embedding

In [ ]:
hidden_layer_nodes = 10
num_features = train_data.vocab_size # The three values in the sting of protein, but this have I made as a array that is onehot encoded
num_output = train_data.vocab_size # The number of of possible values

class CasualSelfAttention(nn.Module):
    def __init__(
        self,
        d,    # Dimension of the embedding 
        H,    # Amount of heads
        T=129,
        bias=False # Length of the input (our proteins are always 129 aminoacids long)
        ):
        super(CasualSelfAttention, self).__init__()
        assert d % H == 0 # the dimension of the embedding has to be clean divisible by the amount of heads 

        # key, query, value projections for all heads, but in a batch
        # output is 3X the dimension because it includes key, query and value
        self.c_attn = nn.Linear(d, 3*d, bias=bias)
        print(f"c_attn: {self.c_attn}")

        # projection of concatenated attention head outputs
        self.c_proj = nn.Linear(d, d, bias=bias)
        print(f"c_proj: {self.c_proj}")


        # input embedding layer
        self.embed = nn.Embedding(num_features, d)
        print(f"Embedding layer: {self.embed}")

        # positional embedding
        self.pos_embed = nn.Embedding(T, d)
        print(f"Positional embedding layer: {self.pos_embed}")

    def forward(self, token_ids):
        # token_ids: (batch_size, sequence_length)
        T = token_ids.shape[1]
        positions = torch.arange(T, device=token_ids.device)

        token_vectors = self.token_embed(token_ids)    # (B, T, d)
        position_vectors = self.position_embed(positions)  # (T, d)

        return token_vectors + position_vectors        # (B, T, d)


embedding = CasualSelfAttention(d=128, H=16, T=129)

x, target, family, length = train_data[0]

#hidden = embedding(x.unsqueeze(0))

#print(hidden.shape)  # torch.Size([1, 129, 128])

c_attn: Linear(in_features=128, out_features=384, bias=False)
Embedding layer: Embedding(23, 128)
Positional embedding layer: Embedding(129, 128)


In [ ]:
class AminoTransformer(nn.Module):
    def __init__(self, vocab_size, d, T, H):
        super().__init__()
        
        # 1. Embeddings go here, at the very beginning of the model
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(T, d)
        
        # 2. A stack of Attention/Transformer blocks
        self.blocks = nn.Sequential(
            CausalSelfAttention(d, H, T),
            CausalSelfAttention(d, H, T),
            CausalSelfAttention(d, H, T),
            # ... usually 12 to 96 of these
        )

    def forward(self, idx):
        # idx is raw token integers of shape [Batch, Time]
        B, T = idx.size()
        
        # Create positional integers (0, 1, 2... T)
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        
        # ADD embeddings together here!
        x = self.token_embedding(idx) + self.position_embedding(pos)
        
        # Pass the combined vectors into the stack of attention blocks
        x = self.blocks(x) 
        return x

In [ ]:
"""
Source: https://github.com/karpathy/nanoGPT/blob/master/model.py
"""


class CausalSelfAttention(nn.Module):

    def __init__(
        self,
        d,
        H,
        T,
        bias=False,
        dropout=0.2,
    ):
        """
        Arguments:
        d: size of embedding dimension
        H: number of attention heads
        T: maximum length of input sequences (in tokens)
        bias: whether or not to use bias in linear layers
        dropout: probability of dropout
        """
        super().__init__()
        assert d % H == 0

        # input embedding layer
        self.embed = nn.Embedding(num_features, d)
        print(f"Embedding layer: {self.embed}")

        # positional embedding
        self.pos_embed = nn.Embedding(T, d)
        print(f"Positional embedding layer: {self.pos_embed}")


        # key, query, value projections for all heads, but in a batch
        # output is 3X the dimension because it includes key, query and value
        self.c_attn = nn.Linear(d, 3*d, bias=bias)
        print(f"c_attn: {self.c_attn}")

        # projection of concatenated attention head outputs
        self.c_proj = nn.Linear(d, d, bias=bias)

        # dropout modules
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.H = H
        self.d = d

        # causal mask to ensure that attention is only applied to
        # the left in the input sequence
        self.register_buffer("mask", torch.tril(torch.ones(T, T))
                                    .view(1, 1, T, T))

    def forward(self, x):
        B, T, _ = x.size() # batch size, sequence length, embedding dimensionality

        # compute query, key, and value vectors for all heads in batch
        # split the output into separate query, key, and value tensors
        q, k, v  = self.c_attn(x).split(self.d, dim=2) # [B, T, d]

        # reshape tensor into sequences of smaller token vectors for each head
        k = k.view(B, T, self.H, self.d // self.H).transpose(1, 2) # [B, H, T, d // H]
        q = q.view(B, T, self.H, self.d // self.H).transpose(1, 2)
        v = v.view(B, T, self.H, self.d // self.H).transpose(1, 2)

        # compute the attention matrix, perform masking, and apply dropout
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1))) # [B, H, T, T]
        att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        # compute output vectors for each token
        y = att @ v # [B, H, T, d // H]

        # concatenate outputs from each attention head and linearly project
        y = y.transpose(1, 2).contiguous().view(B, T, self.d)
        y = self.resid_dropout(self.c_proj(y))
        return y

embedding = CausalSelfAttention(d=128, H=16, T=129)

Embedding layer: Embedding(23, 128)
Positional embedding layer: Embedding(129, 128)
c_attn: Linear(in_features=128, out_features=384, bias=False)
